In [13]:
import ipywidgets as widgets
from ipywidgets import VBox, widgets
from IPython.display import display, HTML
from tensorflow.keras.models import load_model
import sounddevice as sd
import numpy as np
import librosa
from librosa.display import specshow
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2

# sounddevice constants
TARGET_SR = 16000

# gpu
tf.config.set_visible_devices([], 'GPU') # turn off GPUs

In [2]:
from ipywidgets import VBox, HBox, Button, Text, Label, Output, FloatText

# Define widgets
model_path_widget = Text(description="Model Path:")
load_button = Button(description="Load Model")
duration_widget = FloatText(value=1.0, description="Duration (s):")
record_button = Button(description="Record")
preprocess_button = Button(description="Preprocess")
classify_button = Button(description="Classify")

# Define outputs for capturing feedback
output_model = Output()
output_audio = Output()
output_preprocess = Output()
output_classification = Output()

In [3]:
# Model loading widget
def load_model_callback(button):
    with output_model:
        output_model.clear_output()
        try:
            model_path = model_path_widget.value
            global model
            model = load_model(model_path)
            print(f"Model loaded successfully from {model_path}")
        except Exception as e:
            print(f"Error loading model: {e}")

load_button.on_click(load_model_callback)

In [4]:
duration_widget = widgets.FloatText(value=1.0, description="Duration (s):")
record_button = widgets.Button(description="Record")
output_audio = widgets.Output()

def record_audio_callback(button):
    with output_audio:
        output_audio.clear_output()
        try:
            duration = duration_widget.value
            print("Recording...")
            global recorded_audio
            recorded_audio = sd.rec(int(duration * TARGET_SR), samplerate=TARGET_SR, channels=1)
            sd.wait()
            print("Recording complete.")
        except Exception as e:
            print(f"Error recording audio: {e}")

record_button.on_click(record_audio_callback)

In [5]:
def resize_spectrogram(spectrogram, target_shape=(128, 32)):
    """
    Resize spectrogram to match the model's input shape.

    Args:
        spectrogram (np.ndarray): Input spectrogram.
        target_shape (tuple): Target shape (height, width).

    Returns:
        np.ndarray: Resized spectrogram.
    """
    return cv2.resize(spectrogram, target_shape, interpolation=cv2.INTER_AREA)

def generate_spectrogram(segment, sr=16000):
    """
    Generates a Mel spectrogram from an audio segment.

    Args:
        segment (np.ndarray): Audio segment.
        sr (int): Sampling rate in Hz.

    Returns:
        np.ndarray: Mel spectrogram in decibel units.
    """
    S = librosa.feature.melspectrogram(y=segment, sr=sr, n_fft=2048, hop_length=512, n_mels=128)
    return librosa.power_to_db(S, ref=np.max)

def preprocess_callback(button):
    with output_preprocess:
        output_preprocess.clear_output()
        try:
            if 'recorded_audio' not in globals():
                print("No audio recorded.")
                return
            # Generate spectrogram
            spectrogram = generate_spectrogram(recorded_audio.flatten(), sr=TARGET_SR)
            spectrogram = resize_spectrogram(spectrogram, target_shape=(128, 32))
            global spectrogram_chunks
            spectrogram_chunks = np.array_split(spectrogram, spectrogram.shape[1] // model.input_shape[2], axis=1)
            
            # Display spectrogram
            plt.figure(figsize=(10, 4))
            specshow(spectrogram, sr=TARGET_SR, x_axis='time', y_axis='mel')
            plt.title('Mel Spectrogram')
            plt.colorbar(format='%+2.0f dB')
            plt.show()
        except Exception as e:
            print(f"Error preprocessing audio: {e}")

preprocess_button.on_click(preprocess_callback)

In [6]:
def weighted_average(predictions, threshold=0.5):
    """
    Compute a weighted average of predictions.

    Args:
        predictions (list): List of chunk predictions.
        threshold (float): Confidence threshold to give more weight to higher predictions.

    Returns:
        float: Weighted average.
    """
    weights = [max(p - threshold, 0) for p in predictions]  # Penalize values below threshold
    if sum(weights) == 0:
        weights = [1] * len(predictions)  # Default to equal weights if all are below threshold
    weighted_avg = sum(p * w for p, w in zip(predictions, weights)) / sum(weights)
    return weighted_avg

def percentile_aggregation(predictions, percentile=75):
    """
    Aggregate predictions based on the top percentile.

    Args:
        predictions (list): List of chunk predictions.
        percentile (int): Percentile value (e.g., 75).

    Returns:
        float: Aggregated prediction based on the given percentile.
    """
    threshold = np.percentile(predictions, percentile)
    top_predictions = [p for p in predictions if p >= threshold]
    return np.mean(top_predictions)

def remove_silence(audio, sr, top_db=20):
    """
    Remove silence from audio.

    Args:
        audio (np.ndarray): Audio signal.
        sr (int): Sampling rate.
        top_db (int): Threshold (in dB) for silence detection.

    Returns:
        np.ndarray: Audio signal with silence removed.
    """
    intervals = librosa.effects.split(audio, top_db=top_db)
    non_silent_audio = np.concatenate([audio[start:end] for start, end in intervals])
    return non_silent_audio

In [15]:
classify_button = widgets.Button(description="Classify")
output_classification = widgets.Output()

def classify_callback(button):
    with output_classification:
        output_classification.clear_output()
        try:
            if 'spectrogram_chunks' not in globals():
                print("No spectrogram to classify.")
                return
            
            print(f"Number of chunks: {len(spectrogram_chunks)}")
            predictions = []
            
            for i, chunk in enumerate(spectrogram_chunks):
                chunk_resized = resize_spectrogram(chunk, target_shape=(128, 32))
                chunk_resized = np.expand_dims(chunk_resized, axis=(0, -1))
                pred = model.predict(chunk_resized.T)
                predictions.append(pred[0, 0])
                print(f"Chunk {i + 1}: Prediction = {pred[0, 0]:.4f}")
            
            # Aggregation
            weighted_avg = weighted_average(predictions, threshold=0.5)
            percentile_avg = percentile_aggregation(predictions, percentile=75)
            final_class = 1 if weighted_avg > 0.5 else 0
            
            print(f"\nWeighted Average Prediction: {weighted_avg:.4f}")
            print(f"Percentile-based Average Prediction: {percentile_avg:.4f}")
            print(f"Final Classification: {'Unlocked' if final_class == 1 else 'Locked'}")
            
            # Display lock status
            icon = "🔓" if final_class == 1 else "🔒"
            color = "green" if final_class == 1 else "red"
            status_html = f"""
            <div style="text-align:center; font-size:40px; color:{color};">
                {icon} {'Unlocked' if final_class == 1 else 'Locked'}
            </div>
            """
            display(HTML(status_html))
        except Exception as e:
            print(f"Error classifying audio: {e}")

classify_button.on_click(classify_callback)

In [16]:
frontend = VBox([
    Label(value="Voice Lock Simulation"),
    HBox([model_path_widget, load_button]),
    output_model,
    HBox([duration_widget, record_button]),
    output_audio,
    preprocess_button,
    output_preprocess,
    classify_button,
    output_classification
])

display(frontend)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/stepp